## Feature Fetch API

In [1]:
# import shutil

# # Source file path (original name)
# source_path = '/kaggle/input/datasets/sciencekonstant/truecategories/final_categories.json'

# # Destination path including the *new* file name
# destination_path = '/kaggle/working/ordered_categories.json'

# # Copy and rename in one step
# shutil.copy(source_path, destination_path)

'/kaggle/working/ordered_categories.json'

In [2]:
!pip install modal
!modal token set --token-id ak-haLhofOMfl5Sfl35PC3nBe --token-secret as-M0MNOtkQMcbeqqvfm1CzIk --profile=sciencekonstant

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.2/985.2 kB 4.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 18.2 MB/s eta 0:00:00
Verifying token against https://api.modal.com,https://api.modal2.com
Token verified successfully!
⠋ Storing token
Token written to /root/.modal.toml in profile sciencekonstant.


In [3]:
# %%bash
# 1. Create a persistent volume in Modal
# modal volume create overture-data

# 2. Upload your files into the root (/) of that volume
# (Replace the local Kaggle paths with your actual paths!)
# modal volume put overture-data /kaggle/working/ordered_categories.json /
# modal volume put overture-data /kaggle/input/datasets/sciencekonstant/overture/categories.csv /

# echo "✅ Files successfully uploaded to Modal Cloud!"

✅ Files successfully uploaded to Modal Cloud!


Usage: modal volume put [OPTIONS] VOLUME_NAME LOCAL_PATH [REMOTE_PATH]
Try 'modal volume put -h' for help.

Error: /ordered_categories.json: already exists


In [4]:
%%writefile api.py
import modal
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Dict, Any, List
import json

# 1. DEFINE THE HIGH-STANDARD CONTAINER ENVIRONMENT
# We explicitly install and use 'uv' to build the container dependencies lightning-fast
image = (
    modal.Image.debian_slim(python_version="3.11")
    .run_commands("pip install uv")
    # 1. Install all standard packages
    .run_commands("uv pip install --system fastapi[standard] pydantic duckdb geopandas pandas numpy shapely")
    # 2. Install the lightweight CPU-only version of PyTorch
    .run_commands("uv pip install --system torch --index-url https://download.pytorch.org/whl/cpu")
)

app = modal.App("overtureapi")
vol = modal.Volume.from_name("overture-data")

# --- PASTE YOUR ENTIRE ENGINE CODE HERE ---
# (Paste load_taxonomies, OvertureFeatureEngineGPU, fetch_online_context, and get_online_features here)
# Make sure to include all your standard imports (duckdb, geopandas, torch, etc.)
import os
import torch
import duckdb
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkb
import re
import warnings
import gc

# =====================================================================
# 1. TAXONOMY CACHE (Run once on server startup)
# =====================================================================
def load_taxonomies(taxonomy_csv_path):
    df = pd.read_csv(taxonomy_csv_path)
    
    # Clean strings safely to guarantee exact matching
    for col in ['New Primary Category', 'New Basic Level Category', 'Group (L0)']:
        df[col] = df[col].astype(str).str.lower().str.strip()
    
    valid_df = df[(df['New Primary Category'] != 'nan') & (df['New Primary Category'] != '')]
    
    competitor_mappings = {}
    grouped_blc = valid_df.groupby('New Basic Level Category')['New Primary Category'].apply(lambda x: list(set(x))).to_dict()
    
    for blc, categories in grouped_blc.items():
        for cat in categories:
            competitor_mappings[cat] = categories
            
    synergy_mappings = {}
    
    # Bulletproof float filter: forces everything to string and drops empty/nan
    l0_groups = [str(g) for g in valid_df['Group (L0)'].unique() if str(g) not in ('nan', 'None', '')]
    
    for group in l0_groups:
        categories_in_group = valid_df[valid_df['Group (L0)'] == group]['New Primary Category'].tolist()
        synergy_mappings[group] = list(set(categories_in_group))
        
    # Print statements removed for API stability
    return competitor_mappings, synergy_mappings

# =====================================================================
# 2. THE GPU FEATURE ENGINE (Exact replica from Cell 2)
# =====================================================================
class OvertureFeatureEngineGPU:
    def __init__(self, target_crs="EPSG:7755", batch_size=250_000_000):
        """
        EPSG:7755 (India Metric) ensures all distances are in exactly meters.
        batch_size prevents CUDA Out-Of-Memory errors during matrix multiplication.
        """
        self.crs = target_crs
        self.batch_size = batch_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Engine Online. Core computation device: {self.device.type.upper()}")

    def fit(self, overture_dict):
        """
        Loads all Overture themes into the engine's memory.
        Expects a dict: {'places': gdf, 'roads': gdf, 'connectors': gdf, 'base': gdf}
        """
        print("Projecting Overture Themes to Metric System...")

        # Paranoia VRAM flush: guarantees previous block's tensors are destroyed
        torch.cuda.empty_cache()
        
        # Isolate projection memory: pop -> project -> delete -> collect
        tmp = overture_dict.pop('places', gpd.GeoDataFrame())
        self.places = tmp.to_crs(self.crs) if not tmp.empty else tmp
        del tmp; gc.collect()

        tmp = overture_dict.pop('roads', gpd.GeoDataFrame())
        self.roads = tmp.to_crs(self.crs) if not tmp.empty else tmp
        del tmp; gc.collect()

        tmp = overture_dict.pop('connectors', gpd.GeoDataFrame())
        self.connectors = tmp.to_crs(self.crs) if not tmp.empty else tmp
        del tmp; gc.collect()

        tmp = overture_dict.pop('base', gpd.GeoDataFrame())
        self.base = tmp.to_crs(self.crs) if not tmp.empty else tmp
        del tmp; gc.collect()

        # # 1. Pre-load GPU Tensors for blazing fast Point math
        print("Caching Point Clouds to VRAM...")

       # In fit(): Update empty tensor initializations and connector fallback
        self.places_tensor = self._to_tensor(self.places) if not self.places.empty else torch.empty((0,2), device=self.device, dtype=torch.float64)
        self.connectors_tensor = self._to_tensor(self.connectors) if not self.connectors.empty else torch.empty((0,2), device=self.device, dtype=torch.float64)

        # Segment Places by Taxonomy for O(1) filtering
        self.cat_tensors = {}
        # Dynamically find the category column based on Overture release version
        cat_col = next((c for c in ['primary_category', 'categories_primary', 'category', 'subtype'] 
                        if hasattr(self, 'places') and not self.places.empty and c in self.places.columns), None)
        
        if cat_col:
            # Normalize strings safely
            categories = self.places[cat_col].astype(str).str.lower().str.strip()
            for cat in categories.unique():
                mask = categories == cat
                self.cat_tensors[cat] = self._to_tensor(self.places[mask])
                
        if self.connectors_tensor.shape[0] == 0 and hasattr(self, 'roads') and not self.roads.empty:
            print("Connectors missing. Extracting road vertices as proxy junctions...")
            def get_coords(geom):
                if geom is None or geom.is_empty: return []
                # FIX: Force extraction of only X and Y (pt[:2]) to drop rogue Z-elevations
                if geom.geom_type == 'LineString': return [pt[:2] for pt in geom.coords]
                if geom.geom_type == 'MultiLineString': return [pt[:2] for line in geom.geoms for pt in line.coords]
                return []
            
            all_coords = [pt for geom in self.roads.geometry for pt in get_coords(geom)]
            if all_coords:
                # Use float64 to match query tensor dtype
                self.connectors_tensor = torch.tensor(all_coords, dtype=torch.float64, device=self.device)

        print("Engine Fit Complete. Ready for transformation.")
        # FREE HUGE CPU RAM: We don't need point geometries on the CPU anymore!
        if hasattr(self, 'places'): del self.places
        if hasattr(self, 'connectors'): del self.connectors
        gc.collect()

    def _to_tensor(self, gdf):
        """Moves coordinates to GPU natively. Fast-paths Points to skip CPU GEOS overhead."""
        if gdf.empty:
            return torch.empty((0,2), device=self.device, dtype=torch.float64)
            
        # FAST PATH: If geometries are Points, bypass the heavy GEOS engine instantly
        if (gdf.geom_type == 'Point').all():
            coords = np.column_stack((gdf.geometry.x, gdf.geometry.y))
        else:
            # Fallback for Polygons/Lines
            centroids = gdf.geometry.centroid
            coords = np.column_stack((centroids.x, centroids.y))
            
        coords = np.nan_to_num(coords, nan=np.inf)
        return torch.tensor(coords, dtype=torch.float64, device=self.device)

    def _gpu_point_analytics(self, query_tensor, target_tensor, radii=[100, 300, 500, 1000]):
        """Computes multi-radius densities using float64 to ensure meter-level precision."""
        N_queries = query_tensor.shape[0]
        N_targets = target_tensor.shape[0]
        
        if N_targets == 0:
            return {r: np.zeros(N_queries, dtype=int) for r in radii}, \
                   np.full(N_queries, np.nan), np.zeros(N_queries, dtype=float)

        counts = {r: torch.zeros(N_queries, dtype=torch.int32, device=self.device) for r in radii}
        # Downcast outputs back to float32 to conserve memory
        nearest = torch.full((N_queries,), float('nan'), device=self.device, dtype=torch.float32)
        gravity = torch.zeros(N_queries, dtype=torch.float32, device=self.device)

        safe_batch = max(1, self.batch_size // max(1, N_targets))

        for i in range(0, N_queries, safe_batch):
            q_batch = query_tensor[i:i+safe_batch]
            
            # cdist automatically uses float64 here, restoring 1-meter precision
            dist = torch.cdist(q_batch, target_tensor)
            
            # Radii Counts: (dist > 1e-4) strictly ignores the query point matching itself
            for r in radii:
                counts[r][i:i+safe_batch] = ((dist > 1e-4) & (dist <= r)).sum(dim=1).to(torch.int32)
                
            if target_tensor.shape[0] > 1:
                top2_dist, _ = torch.topk(dist, k=2, dim=1, largest=False)
                is_self = top2_dist[:, 0] < 1e-4
                min_d = torch.where(is_self, top2_dist[:, 1], top2_dist[:, 0])
            elif target_tensor.shape[0] == 1:
                min_d = torch.where(dist[:, 0] < 1e-4, torch.tensor(float('nan'), device=self.device, dtype=torch.float64), dist[:, 0])
            else:
                min_d = torch.full((q_batch.shape[0],), float('nan'), device=self.device, dtype=torch.float64)

            min_d[min_d == float('inf')] = float('nan')
            nearest[i:i+safe_batch] = min_d.to(torch.float32)
            
            # Gravity Decay evaluates perfectly now with true meter distances
            decay = 1.0 / ((dist / 100.0)**2 + 1.0)
            decay[dist > 2000] = 0.0 
            gravity[i:i+safe_batch] = decay.sum(dim=1).to(torch.float32)
            
            # Instantly free the massive contiguous blocks back to the GPU pool
            del q_batch, dist, decay

        return {r: counts[r].cpu().numpy() for r in radii}, nearest.cpu().numpy(), gravity.cpu().numpy()

    def transform(self, query_batch, taxonomies, competitor_mappings=None):
        """Generates ML Features, mapping direct categories and synergies."""
        if query_batch.crs is None:
            raise ValueError("query_batch must have a defined CRS before processing.")
            
        targets = query_batch.to_crs(self.crs).copy()
        q_tensor = self._to_tensor(targets)
        
        # Normalize target categories to guarantee matching with dictionary keys
        if 'target_category' in targets.columns:
            targets['target_category'] = targets['target_category'].astype(str).str.lower().str.strip()
        else:
            targets['target_category'] = 'unknown'
        
        # 1. SCHEMA LOCK: Penalties strictly aligned to the 10000m max_distance threshold
        targets['comp_count_100m'] = 0
        targets['comp_count_300m'] = 0
        targets['comp_count_1000m'] = 0
        targets['comp_count_5000m'] = 0
        targets['nearest_comp_dist'] = 10000.0  
        targets['comp_gravity_score'] = 0.0
        
        for group in taxonomies.keys():
            targets[f'synergy_{group}_300m'] = 0
            targets[f'synergy_{group}_500m'] = 0
            targets[f'synergy_{group}_1000m'] = 0
            
        targets['junction_density_300m'] = 0
        targets['corner_lot_indicator'] = 0
        
        targets['nearest_road_class'] = 'unclassified'
        targets['nearest_road_surface'] = 'unknown'
        targets['dist_nearest_water'] = 10000.0 
        targets['dist_nearest_park'] = 10000.0

        # 2. GROUP 1: Direct Competition (Exact Set Intersection)
        for cat in targets['target_category'].unique():
            mask = targets['target_category'] == cat
            cat_q_tensor = self._to_tensor(targets[mask])
            clean_cat = str(cat).lower().strip()
            
            # 1. Get the bucket of true competitors
            bucket_list = competitor_mappings.get(clean_cat, [clean_cat]) if competitor_mappings else [clean_cat]
            bucket_set = set([str(c).lower().strip() for c in bucket_list])
            
            # 2. EXACT INTERSECTION: Prevents "bar" from matching "barber_shop" 
            # while safely handling stringified arrays like "['cafe', 'bakery']"
           # 2. EXACT INTERSECTION
            matched_keys = [
                k for k in self.cat_tensors.keys() 
                if set(re.findall(r'[a-z_]+', str(k))).intersection(bucket_set)
            ]
            
            tensors = [self.cat_tensors[k] for k in matched_keys]
            t_tensor = torch.cat(tensors, dim=0) if tensors else torch.empty((0,2), device=self.device, dtype=torch.float64)
            
            counts, nearest, grav = self._gpu_point_analytics(cat_q_tensor, t_tensor, radii=[100, 300, 1000, 5000])
            
            targets.loc[mask, 'comp_count_100m'] = counts[100]
            targets.loc[mask, 'comp_count_300m'] = counts[300]
            targets.loc[mask, 'comp_count_1000m'] = counts[1000]
            targets.loc[mask, 'comp_count_5000m'] = counts[5000]
            
            targets.loc[mask, 'nearest_comp_dist'] = np.nan_to_num(nearest, nan=10000.0)
            targets.loc[mask, 'comp_gravity_score'] = grav

        # 3. GROUP 2: Synergies (Exact Set Intersection)
        for group_name, cat_list in taxonomies.items():
            bucket_set = set([str(c).lower().strip() for c in cat_list])
            
            matched_keys = [
                k for k in self.cat_tensors.keys() 
                if set(re.findall(r'[a-z_]+', str(k))).intersection(bucket_set)
            ]
            
            tensors = [self.cat_tensors[k] for k in matched_keys]
            
            if tensors:
                combined_target = torch.cat(tensors, dim=0)
                counts, _, _ = self._gpu_point_analytics(q_tensor, combined_target, radii=[300, 500, 1000])
                targets[f'synergy_{group_name}_300m'] = counts[300]
                targets[f'synergy_{group_name}_500m'] = counts[500]
                targets[f'synergy_{group_name}_1000m'] = counts[1000]


       # 4. GROUP 3: Traffic & Morphology
        if self.connectors_tensor.shape[0] > 0:
            conn_counts, _, _ = self._gpu_point_analytics(q_tensor, self.connectors_tensor, radii=[25, 300])
            targets['junction_density_300m'] = conn_counts[300]
            targets['corner_lot_indicator'] = (conn_counts[25] > 0).astype(int) 

        # 5. GROUP 4 & 5: Environment & Roads
        if hasattr(self, 'roads') and not self.roads.empty:
            r_class_col = next((c for c in ['class', 'highway', 'type', 'subtype'] if c in self.roads.columns), None)
            r_surf_col = next((c for c in ['surface', 'road_surface'] if c in self.roads.columns), None)
            
            keep_cols = ['geometry']
            if r_class_col: keep_cols.append(r_class_col)
            if r_surf_col: keep_cols.append(r_surf_col)
                
            r_near = gpd.sjoin_nearest(targets[['geometry']], self.roads[keep_cols], how='left', max_distance=10000)
            r_near = r_near[~r_near.index.duplicated(keep='first')]
        
            # NEW CODE
            if r_class_col and r_class_col in r_near.columns:
                targets['nearest_road_class'] = r_near[r_class_col].astype(object).fillna('unclassified').astype(str)
            if r_surf_col and r_surf_col in r_near.columns:
                targets['nearest_road_surface'] = r_near[r_surf_col].astype(object).fillna('unknown').astype(str)

        if hasattr(self, 'base') and not self.base.empty and 'subtype' in self.base.columns:
            # CPU SAVER: Vectorized exact column search instead of slow row-by-row string concatenation
            subtypes = self.base['subtype'].astype(str).str.lower()
            
            water = self.base[subtypes.str.contains('water|river|lake|pond|ocean|stream')]
            parks = self.base[subtypes.str.contains('park|forest|nature|grass|wood|meadow')]
            
            if not water.empty:
                w_near = gpd.sjoin_nearest(targets[['geometry']], water, how='left', distance_col='d_water', max_distance=10000)
                w_near = w_near[~w_near.index.duplicated(keep='first')]
                targets['dist_nearest_water'] = w_near['d_water'].fillna(10000.0)
            
            if not parks.empty:
                p_near = gpd.sjoin_nearest(targets[['geometry']], parks, how='left', distance_col='d_park', max_distance=10000)
                p_near = p_near[~p_near.index.duplicated(keep='first')]
                targets['dist_nearest_park'] = p_near['d_park'].fillna(10000.0)
        
        return targets.to_crs("EPSG:4326")

# =====================================================================
# 3. PRODUCTION API MATERLIAZER (Sub-Second R-Tree Querying)
# =====================================================================
def fetch_online_context(lat, lon, db_path="india_spatial.db", max_distance_meters=5500):
    buffer_deg = (max_distance_meters / 111_000)
    minx, maxx = lon - buffer_deg, lon + buffer_deg
    miny, maxy = lat - buffer_deg, lat + buffer_deg
    
    # Connect to the indexed database in read-only mode for maximum speed
    con = duckdb.connect(db_path, read_only=True)
    con.execute("INSTALL spatial; LOAD spatial;")
    
    bbox_geom = f"ST_MakeEnvelope({minx}, {miny}, {maxx}, {maxy})"
    
    def query_table(table_name, cols):
        select_cols = ", ".join(cols) + "," if cols else ""
        
        # DuckDB automatically detects and uses the R-Tree index here
        query = f"""
            SELECT {select_cols} ST_AsWKB(geometry) as geometry
            FROM {table_name}
            WHERE ST_Intersects(geometry, {bbox_geom})
        """
        
        df = con.execute(query).df()
        
        if df.empty:
            return gpd.GeoDataFrame(columns=cols + ['geometry'], geometry='geometry', crs="EPSG:4326")
            
        safe_wkb = [bytes(b) if b is not None else None for b in df['geometry']]
        return gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkb(safe_wkb), crs="EPSG:4326")

    # Query the indexed tables directly
    places_gdf = query_table("places", ["primary_category"])
    roads_gdf = query_table("roads", ["class", "surface"])
    conn_gdf = query_table("connectors", [])
    base_gdf = query_table("base", ["subtype"])
    
    con.close()
    
    return {
        'places': places_gdf, 'roads': roads_gdf,
        'connectors': conn_gdf, 'base': base_gdf
    }

# =====================================================================
# 4. THE INFERENCE PIPELINE API
# =====================================================================
def get_online_features(lat, lon, target_category, engine, competitor_mappings, synergy_mappings, db_path):
    # Pass the db_path string instead of the dictionary
    overture_context = fetch_online_context(lat, lon, db_path=db_path, max_distance_meters=5500)
    
    query_gdf = gpd.GeoDataFrame(
        [{'target_category': target_category}], 
        geometry=gpd.points_from_xy([lon], [lat]), 
        crs="EPSG:4326"
    )
    
    engine.fit(overture_context)
    features_gdf = engine.transform(query_gdf, synergy_mappings, competitor_mappings)
    
    features_df = pd.DataFrame(features_gdf.drop(columns=['geometry']))
    return features_df.iloc[0].to_dict()

# =====================================================================
# 5. NEW VECTORIZED BATCH ENGINE (Does not modify existing logic)
# =====================================================================
def get_all_category_features_vectorized(lat, lon, engine, ordered_categories, competitor_mappings, synergy_mappings, db_path):
    """Computes shared context once and vectorizes competitor distances for the master list."""
    
    # 1. Fetch and fit context normally
    overture_context = fetch_online_context(lat, lon, db_path=db_path, max_distance_meters=5500)
    engine.fit(overture_context)

    # 2. Project single coordinate for shared features
    query_gdf = gpd.GeoDataFrame(geometry=gpd.points_from_xy([lon], [lat]), crs="EPSG:4326")
    q_proj = query_gdf.to_crs(engine.crs)
    q_tensor = engine._to_tensor(q_proj)

    # Pre-fill shared features
    shared = {
        'nearest_road_class': 'unclassified',
        'nearest_road_surface': 'unknown',
        'dist_nearest_water': 10000.0,
        'dist_nearest_park': 10000.0,
        'junction_density_300m': 0,
        'corner_lot_indicator': 0
    }
    
    for group in synergy_mappings.keys():
        shared[f'synergy_{group}_300m'] = 0
        shared[f'synergy_{group}_500m'] = 0
        shared[f'synergy_{group}_1000m'] = 0

    # -- Calculate Roads & Environment --
    if hasattr(engine, 'roads') and not engine.roads.empty:
        r_class_col = next((c for c in ['class', 'highway', 'type', 'subtype'] if c in engine.roads.columns), None)
        r_surf_col = next((c for c in ['surface', 'road_surface'] if c in engine.roads.columns), None)
        
        keep_cols = ['geometry']
        if r_class_col: keep_cols.append(r_class_col)
        if r_surf_col: keep_cols.append(r_surf_col)
            
        r_near = gpd.sjoin_nearest(q_proj[['geometry']], engine.roads[keep_cols], how='left', max_distance=10000)
        r_near = r_near[~r_near.index.duplicated(keep='first')]

        if r_class_col and r_class_col in r_near.columns:
            shared['nearest_road_class'] = str(r_near[r_class_col].astype(object).fillna('unclassified').iloc[0])
        if r_surf_col and r_surf_col in r_near.columns:
            shared['nearest_road_surface'] = str(r_near[r_surf_col].astype(object).fillna('unknown').iloc[0])

    if hasattr(engine, 'base') and not engine.base.empty and 'subtype' in engine.base.columns:
        subtypes = engine.base['subtype'].astype(str).str.lower()
        
        water = engine.base[subtypes.str.contains('water|river|lake|pond|ocean|stream')]
        parks = engine.base[subtypes.str.contains('park|forest|nature|grass|wood|meadow')]
        
        if not water.empty:
            w_near = gpd.sjoin_nearest(q_proj[['geometry']], water, how='left', distance_col='d_water', max_distance=10000)
            w_near = w_near[~w_near.index.duplicated(keep='first')]
            shared['dist_nearest_water'] = round(float(w_near['d_water'].fillna(10000.0).iloc[0]), 2)
        
        if not parks.empty:
            p_near = gpd.sjoin_nearest(q_proj[['geometry']], parks, how='left', distance_col='d_park', max_distance=10000)
            p_near = p_near[~p_near.index.duplicated(keep='first')]
            shared['dist_nearest_park'] = round(float(p_near['d_park'].fillna(10000.0).iloc[0]), 2)

    # -- Calculate Junctions --
    if engine.connectors_tensor.shape[0] > 0:
        conn_counts, _, _ = engine._gpu_point_analytics(q_tensor, engine.connectors_tensor, radii=[25, 300])
        shared['junction_density_300m'] = int(conn_counts[300][0])
        shared['corner_lot_indicator'] = 1 if conn_counts[25][0] > 0 else 0

    # -- Calculate Synergies --
    for group_name, cat_list in synergy_mappings.items():
        bucket_set = set([str(c).lower().strip() for c in cat_list])
        matched_keys = [k for k in engine.cat_tensors.keys() if set(re.findall(r'[a-z_]+', str(k))).intersection(bucket_set)]
        tensors = [engine.cat_tensors[k] for k in matched_keys]
        
        if tensors:
            combined_target = torch.cat(tensors, dim=0)
            counts, _, _ = engine._gpu_point_analytics(q_tensor, combined_target, radii=[300, 500, 1000])
            shared[f'synergy_{group_name}_300m'] = int(counts[300][0])
            shared[f'synergy_{group_name}_500m'] = int(counts[500][0])
            shared[f'synergy_{group_name}_1000m'] = int(counts[1000][0])

    # -- Pre-compute Distances to ALL Local Places --
    local_dists = {}
    for cat_name, cat_tensor in engine.cat_tensors.items():
        if cat_tensor.shape[0] > 0:
            local_dists[cat_name] = torch.cdist(q_tensor, cat_tensor)[0] # Extracts to 1D tensor
            
    competitor_matrix = []
    
    # -- Vectorized Competitors (Iterates over your 1,591 list instantly) --
    for target_cat in ordered_categories:
        clean_cat = str(target_cat).lower().strip()
        bucket_list = competitor_mappings.get(clean_cat, [clean_cat]) if competitor_mappings else [clean_cat]
        bucket_set = set([str(c).lower().strip() for c in bucket_list])
        
        matched_keys = [k for k in local_dists.keys() if set(re.findall(r'[a-z_]+', str(k))).intersection(bucket_set)]
        
        if not matched_keys:
            # Format: [100m, 300m, 1000m, 5000m, nearest_dist, gravity]
            competitor_matrix.append([0, 0, 0, 0, 10000.0, 0.0])
            continue
            
        dists = torch.cat([local_dists[k] for k in matched_keys])
        dists = dists[dists > 1e-4] # Safely ignores the exact query coordinate (self-matching)
        
        if dists.numel() == 0:
             competitor_matrix.append([0, 0, 0, 0, 10000.0, 0.0])
             continue
             
        c_100 = (dists <= 100).sum().item()
        c_300 = (dists <= 300).sum().item()
        c_1000 = (dists <= 1000).sum().item()
        c_5000 = (dists <= 5000).sum().item()
        nearest = round(dists.min().item(), 2)
        
        decay = 1.0 / ((dists / 100.0)**2 + 1.0)
        decay[dists > 2000] = 0.0
        grav = round(decay.sum().item(), 4)
        
        competitor_matrix.append([c_100, c_300, c_1000, c_5000, nearest, grav])
        
    return shared, competitor_matrix


# ------------------------------------------
# END OF MAIN CODE
# ------------------------------------------

# ---------------------------------------------------------
# 3. FASTAPI SCHEMAS (Production Standard)
# ---------------------------------------------------------
class FeatureRequest(BaseModel):
    lat: float = Field(..., example=28.6304, description="Latitude of the target location")
    lon: float = Field(..., example=77.2177, description="Longitude of the target location")
    category: str = Field(..., example="cafe", description="The business category taxonomy tag")

class FeatureResponse(BaseModel):
    status: str
    features: Dict[str, Any]

class BulkFeatureRequest(BaseModel):
    lat: float = Field(..., example=28.6304, description="Latitude of the target location")
    lon: float = Field(..., example=77.2177, description="Longitude of the target location")

class BulkFeatureResponse(BaseModel):
    status: str
    shared_environment: Dict[str, Any]
    competitors_array: List[List[float]] # The ordered payload containing the 6 competitor metrics

# ---------------------------------------------------------
# 4. THE MODAL CLASS-BASED INFERENCE ENDPOINT
# ---------------------------------------------------------
@app.cls(image=image, volumes={"/data": vol}, cpu=2.0)
class featureAPI:
    @modal.enter()
    def startup(self):
        """
        Lifecycle Hook: Runs exactly ONCE when the container boots up.
        Prevents reloading the DB and model weights on every single user request.
        """
        print("[STARTUP] Booting ML Engine & Loading Taxonomies...")
        self.comp_map, self.syn_map = load_taxonomies('/data/categories.csv')
        self.engine = OvertureFeatureEngineGPU(target_crs="EPSG:7755")
        self.db_path = '/data/india_spatial.db'

        # --- NEW CODE: Load the master ordered list ---
        with open('/data/ordered_categories.json', 'r') as f:
            self.ordered_categories = json.load(f)
            
        print(f"[STARTUP] Engine is warm. Tracking {len(self.ordered_categories)} batch categories.")

    @modal.fastapi_endpoint(method="POST")
    def predict(self, request: FeatureRequest) -> FeatureResponse:
        """
        The live POST endpoint. Takes strict JSON, outputs JSON.
        Includes robust HTTP status code routing for production.
        """
        # 1. 400 BAD REQUEST: Geographic Bounds Check
        # Fast-fail if the coordinates are outside the rough bounding box of India.
        if not (6.0 <= request.lat <= 36.0 and 68.0 <= request.lon <= 98.0):
            raise HTTPException(
                status_code=400, 
                detail="Coordinates out of bounds. This API only supports geographic locations within India."
            )

        try:
            features = get_online_features(
                lat=request.lat, 
                lon=request.lon, 
                target_category=request.category, 
                engine=self.engine, 
                competitor_mappings=self.comp_map, 
                synergy_mappings=self.syn_map, 
                db_path=self.db_path
            )
            
            # 2. 404 NOT FOUND: No features could be calculated (e.g. empty desert)
            if not features or len(features) == 0:
                raise HTTPException(
                    status_code=404, 
                    detail="No geospatial context found for these coordinates."
                )

            return FeatureResponse(status="success", features=features)
            
        except HTTPException:
            # Let our manually raised 400 and 404 exceptions pass through cleanly
            raise
            
        except ValueError as ve:
            # 3. 422 UNPROCESSABLE ENTITY: Math/Logic error inside the engine
            raise HTTPException(status_code=422, detail=f"Data processing error: {str(ve)}")
            
        except Exception as e:
            # 4. 500 INTERNAL SERVER ERROR: Catch-all for CUDA/DuckDB system crashes
            raise HTTPException(status_code=500, detail=f"Internal Server Error: {str(e)}")

    # # --- NEW CODE: The Bulk Endpoint ---
    # @modal.fastapi_endpoint(method="POST", path="/predict_all")
    # def predict_all(self, request: BulkFeatureRequest) -> BulkFeatureResponse:
    
    # --- NEW CODE: The Bulk Endpoint ---
    @modal.fastapi_endpoint(method="POST")
    def predict_all(self, request: BulkFeatureRequest) -> BulkFeatureResponse:
        """
        Evaluates the entire ordered_categories.json master list for a single coordinate in one pass.
        Returns shared context as a dictionary, and competitors as an indexed array.
        """
        if not (6.0 <= request.lat <= 36.0 and 68.0 <= request.lon <= 98.0):
            raise HTTPException(
                status_code=400, 
                detail="Coordinates out of bounds. This API only supports geographic locations within India."
            )

        try:
            shared, comp_matrix = get_all_category_features_vectorized(
                lat=request.lat, 
                lon=request.lon, 
                engine=self.engine,
                ordered_categories=self.ordered_categories,
                competitor_mappings=self.comp_map,
                synergy_mappings=self.syn_map,
                db_path=self.db_path
            )
            
            return BulkFeatureResponse(
                status="success", 
                shared_environment=shared, 
                competitors_array=comp_matrix
            )
            
        except Exception as e:
            raise HTTPException(status_code=500, detail=f"Internal Server Error: {str(e)}")

Writing api.py


## Deployed Feature API

In [5]:
# !modal serve api.py

In [ ]:
# !modal deploy api.py

## Model API

In [4]:
# import shutil

# # Source file path (original name)
# source_path = '/kaggle/input/datasets/sciencekonstant/truecategories/final_categories.json'

# # Destination path including the *new* file name
# destination_path = '/kaggle/working/ordered_categories.json'

# # Copy and rename in one step
# shutil.copy(source_path, destination_path)

In [5]:
!pip install modal
!modal token set --token-id ak-YFWJDq6FGEsfAZuJGVD3XH --token-secret as-sEihb2rtIaVCQIHtSekAeL --profile=shivam-kumar-101075

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 985.2/985.2 kB 14.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 15.7 MB/s eta 0:00:00
Verifying token against https://api.modal.com,https://api.modal2.com
Token verified successfully!
⠋ Storing token
Token written to /root/.modal.toml in profile shivam-kumar-101075.


In [6]:
# %%bash

# modal volume create viability-models

# modal volume put viability-models /kaggle/input/datasets/sciencekonstant/donemodelling/xgb_model.json /
# modal volume put viability-models /kaggle/input/datasets/sciencekonstant/donemodelling/viability_encoder.pkl /
# modal volume put viability-models /kaggle/working/ordered_categories.json /

In [7]:
%%writefile model_api.py
import modal
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from fastapi import HTTPException
from pydantic import BaseModel, Field
from typing import Dict, Any, List, Optional

# 1. CONTAINER IMAGE SETUP
image = (
    modal.Image.debian_slim(python_version="3.11")
    .run_commands("pip install uv")
    .run_commands("uv pip install --system fastapi[standard] pydantic pandas numpy xgboost category_encoders joblib scikit-learn")
)

app = modal.App("viability-model-api")

# --- THIS IS THE ONLY LINE THAT CHANGES ---
vol = modal.Volume.from_name("viability-models")

# =====================================================================
# 2. FEATURE ENGINEERING & PREPROCESSING PIPELINE
# =====================================================================
def run_feature_engineering(df: pd.DataFrame, encoder, feature_names: Optional[List[str]] = None) -> pd.DataFrame:
    df_out = df.copy()
    eps = 1e-5

    # 1. Custom Handcrafted Features
    df_out['commercial_hub_index'] = df_out['junction_density_300m'] * df_out['synergy_services_and_business_1000m']
    df_out['isolation_penalty'] = df_out['nearest_comp_dist'] / (df_out['junction_density_300m'] + 1)

    # 2. Automated Feature Engineering 
    df_out['autoFE_f_3'] = df_out['synergy_services_and_business_300m'] * df_out['synergy_lodging_1000m']
    df_out['autoFE_f_4'] = df_out['isolation_penalty'] - df_out['synergy_cultural_and_historic_300m']
    df_out['autoFE_f_8'] = df_out['synergy_services_and_business_500m'] * df_out['synergy_geographic_entities_1000m']
    df_out['autoFE_f_9'] = df_out['synergy_lifestyle_services_1000m'] * df_out['junction_density_300m']

    df_out['autoFE_f_2'] = df_out['synergy_services_and_business_500m'] / (df_out['synergy_lodging_300m'] + eps)
    df_out['autoFE_f_6'] = df_out['synergy_lifestyle_services_1000m'] / (df_out['synergy_community_and_government_300m'] + eps)
    df_out['autoFE_f_7'] = df_out['synergy_food_and_drink_500m'] / (df_out['synergy_travel_and_transportation_500m'] + eps)
    df_out['autoFE_f_10'] = df_out['synergy_services_and_business_1000m'] / (df_out['synergy_sports_and_recreation_500m'] + eps)
    df_out['autoFE_f_12'] = df_out['synergy_geographic_entities_1000m'] / (df_out['synergy_sports_and_recreation_1000m'] + eps)
    df_out['autoFE_f_13'] = df_out['synergy_food_and_drink_1000m'] / (df_out['dist_nearest_park'] + eps)
    df_out['autoFE_f_14'] = df_out['synergy_lifestyle_services_1000m'] / (df_out['synergy_lodging_1000m'] + eps)

    # 3. Categorical Column Handling
    if 'nearest_road_class' in df_out.columns:
        df_out['nearest_road_class'] = df_out['nearest_road_class'].astype('category')

    # 4. DROP COLUMNS FIRST (Crucial fix for CatBoostEncoder!)
    columns_to_drop = [
        'geometry', '__index_level_0__', 'lat', 'long', 
        'nearest_road_surface', 'viability_score_0_1'
    ]
    df_out = df_out.drop(columns=[c for c in columns_to_drop if c in df_out.columns], errors='ignore')

    # 5. NOW Target Encode (Will receive exactly 64 columns)
    df_out = encoder.transform(df_out)

    # 6. Ensure Exact Column Order for XGBoost
    if feature_names:
        # Fills in any weirdly missing columns with 0.0 to prevent hard crashes
        for col in feature_names:
            if col not in df_out.columns:
                df_out[col] = 0.0
        # Reorders columns to perfectly match the Booster
        df_out = df_out[feature_names]

    return df_out
    
# =====================================================================
# 3. REQUEST & RESPONSE SCHEMAS
# =====================================================================
class SinglePredictRequest(BaseModel):
    # Directly takes the "features" object returned by your spatial /predict endpoint
    features: Dict[str, Any]

class SinglePredictResponse(BaseModel):
    status: str
    target_category: str
    viability_score: float

class BulkPredictRequest(BaseModel):
    # Directly accepts the full payload returned by your spatial /predict_all endpoint
    shared_environment: Dict[str, Any]
    competitors_array: List[List[float]]

class CategoryScore(BaseModel):
    category: str
    viability_score: float

class BulkPredictResponse(BaseModel):
    status: str
    scores: List[float] # Strictly ordered matching ordered_categories.json
    # top_5: List[CategoryScore]

# =====================================================================
# 4. MODAL ENDPOINT CLASS
# =====================================================================
@app.cls(image=image, volumes={"/data": vol}, cpu=2.0)
class ViabilityModelAPI:
    @modal.enter()
    def startup(self):
        """Loads the XGBoost model, CatBoost encoder, and category map into RAM once."""
        print("[STARTUP] Loading Model, Encoder, and Category Index...")
        
        # 1. Load CatBoost Target Encoder
        self.encoder = joblib.load('/data/viability_encoder.pkl')

        # 2. Load XGBoost Model
        self.model = xgb.Booster()
        self.model.load_model('/data/xgb_model.json')
        self.feature_names = self.model.feature_names

        # 3. Load Master Category List
        with open('/data/ordered_categories.json', 'r') as f:
            self.ordered_categories = json.load(f)

        print(f"[STARTUP] Model ready. Loaded {len(self.feature_names or [])} features and {len(self.ordered_categories)} categories.")

    @modal.fastapi_endpoint(method="POST")
    def predict_single(self, request: SinglePredictRequest) -> SinglePredictResponse:
        """Computes viability score for a single category."""
        try:
            df = pd.DataFrame([request.features])
            category_name = str(df.get('target_category', ['unknown'])[0])

            # Apply full feature preprocessing
            X = run_feature_engineering(df, self.encoder, self.feature_names)

            # Predict
            dmatrix = xgb.DMatrix(X, enable_categorical=True)
            score = float(self.model.predict(dmatrix)[0])

            # Clip score to valid target boundaries [0, 1]
            score = float(np.clip(score, 0.0, 1.0))

            return SinglePredictResponse(
                status="success",
                target_category=category_name,
                # viability_score=round(score, 4)
                viability_score=float(score)
            )
        except Exception as e:
            raise HTTPException(status_code=500, detail=f"Inference error: {str(e)}")

    @modal.fastapi_endpoint(method="POST")
    def predict_bulk(self, request: BulkPredictRequest) -> BulkPredictResponse:
        """
        Takes the raw JSON output of /predict_all, vectorizes it across all 1591 categories,
        and returns scores in under 50 milliseconds.
        """
        try:
            comp_cols = [
                'comp_count_100m', 'comp_count_300m', 
                'comp_count_1000m', 'comp_count_5000m', 
                'nearest_comp_dist', 'comp_gravity_score'
            ]
            
            # Fast vectorized dataframe creation (No slow Python row-loops)
            df = pd.DataFrame(request.competitors_array, columns=comp_cols)
            df['target_category'] = self.ordered_categories

            # Broadcast shared environment variables across all rows
            for col, val in request.shared_environment.items():
                df[col] = val

            # Apply full feature preprocessing
            X = run_feature_engineering(df, self.encoder, self.feature_names)

            # High-speed batch prediction
            dmatrix = xgb.DMatrix(X, enable_categorical=True)
            preds = self.model.predict(dmatrix)
            preds = np.clip(preds, 0.0, 1.0).tolist()
            # rounded_scores = [round(float(s), 4) for s in preds]
            exact_scores = [float(s) for s in preds]

            # # Extract Top 5 Viable Categories for convenience
            # paired = sorted(
            #     zip(self.ordered_categories, rounded_scores), 
            #     key=lambda x: x[1], 
            #     reverse=True
            # )
            # top_5 = [{"category": cat, "viability_score": score} for cat, score in paired[:5]]

            return BulkPredictResponse(
                status="success",
                scores=exact_scores
                # top_5=top_5
            )
        except Exception as e:
            raise HTTPException(status_code=500, detail=f"Bulk inference error: {str(e)}")

Overwriting model_api.py


## DEPLOYED THE XGBOOST MODEL

In [8]:
# !modal serve model_api.py

In [9]:
!modal deploy model_api.py

⠸ Creating objects.....
⠦ Creating objects...ggle/working/model_api.py: Uploaded 0/1 missing files
⠏ Creating objects...ggle/working/model_api.py: Finalizing index of 1 files
├── 🔨 Created mount /kaggle/working/model_api.py
├── 🔨 Created function ViabilityModelAPI.*.
├── 🔨 Created Web Function URL for ViabilityModelAPI.predict_bulk => 
│   https://shivam-kumar-101075--viability-model-api-viabilitymodela-0304a7.moda
│   l.run (label truncated)
└── 🔨 Created Web Function URL for ViabilityModelAPI.predict_single => 
    https://shivam-kumar-101075--viability-model-api-viabilitymodela-9ec6ed.moda
⠹ Creating objects...
├── 🔨 Created mount /kaggle/working/model_api.py
├── 🔨 Created function ViabilityModelAPI.*.
├── 🔨 Created Web Function URL for ViabilityModelAPI.predict_bulk => 
│   https://shivam-kumar-101075--viability-model-api-viabilitymodela-0304a7.moda
│   l.run (label truncated)
└── 🔨 Created Web Function URL for ViabilityModelAPI.predict_single => 
    https://shivam-kumar-101075--v